In [1]:
import struct

# ── Pas deze twee dingen aan ──────────────────────────────────────────────────

invoer  = "Werktuig.ply"
uitvoer = "Werktuig_float.ply"

# ─────────────────────────────────────────────────────────────────────────────

with open(invoer, 'rb') as f:
    header_bytes = b''
    while True:
        lijn = f.readline()
        header_bytes += lijn
        if lijn.strip() == b'end_header':
            break

    header_tekst  = header_bytes.decode('utf-8', errors='replace')
    properties    = []
    nieuwe_regels = []
    aantal_punten = 0

    for lijn in header_tekst.split('\n'):
        s = lijn.strip()
        if s.startswith('element vertex'):
            aantal_punten = int(s.split()[-1])
            nieuwe_regels.append(lijn)
        elif s.startswith('property double'):
            naam = s.split()[2]
            properties.append(('double', naam))
            nieuwe_regels.append(lijn.replace('property double', 'property float'))
        elif s.startswith('property float'):
            naam = s.split()[2]
            properties.append(('float', naam))
            nieuwe_regels.append(lijn)
        elif s.startswith('property uchar'):
            naam = s.split()[2]
            properties.append(('uchar', naam))
            nieuwe_regels.append(lijn)
        else:
            nieuwe_regels.append(lijn)

    nieuwe_header = '\n'.join(nieuwe_regels).encode('utf-8')
    stride = sum(8 if t == 'double' else 4 if t == 'float' else 1
                 for t, _ in properties)
    data = f.read()

print(f"Punten: {aantal_punten}")
print(f"Properties: {properties}")

# Bereken automatisch de offset uit de eerste 100 punten
min_x = min_y = min_z = float('inf')
offset = 0
steekproef = min(100, aantal_punten)

for i in range(steekproef):
    for type_, naam in properties:
        if type_ == 'double':
            waarde = struct.unpack_from('<d', data, offset)[0]
            if naam == 'x': min_x = min(min_x, waarde)
            if naam == 'y': min_y = min(min_y, waarde)
            if naam == 'z': min_z = min(min_z, waarde)
            offset += 8
        elif type_ == 'float':
            offset += 4
        elif type_ == 'uchar':
            offset += 1

# Rond af naar gehele getallen voor nette offset
offset_x = int(min_x) if min_x != float('inf') else 0.0
offset_y = int(min_y) if min_y != float('inf') else 0.0
offset_z = int(min_z) if min_z != float('inf') else 0.0

print(f"Automatische verschuiving: ({offset_x}, {offset_y}, {offset_z})")

with open(uitvoer, 'wb') as f:
    f.write(nieuwe_header)

    offset = 0
    for i in range(aantal_punten):
        for type_, naam in properties:
            if type_ == 'double':
                waarde = struct.unpack_from('<d', data, offset)[0]
                if naam == 'x': waarde -= offset_x
                if naam == 'y': waarde -= offset_y
                if naam == 'z': waarde -= offset_z
                f.write(struct.pack('<f', float(waarde)))
                offset += 8
            elif type_ == 'float':
                waarde = struct.unpack_from('<f', data, offset)[0]
                f.write(struct.pack('<f', waarde))
                offset += 4
            elif type_ == 'uchar':
                f.write(bytes([data[offset]]))
                offset += 1

        if i % 500000 == 0:
            print(f"Voortgang: {i}/{aantal_punten} ({100*i//aantal_punten}%)")

print(f"Klaar! Opgeslagen als: {uitvoer}")


Punten: 39672466
Properties: [('float', 'x'), ('float', 'y'), ('float', 'z'), ('float', 'nx'), ('float', 'ny'), ('float', 'nz'), ('uchar', 'red'), ('uchar', 'green'), ('uchar', 'blue'), ('uchar', 'alpha')]
Automatische verschuiving: (0.0, 0.0, 0.0)
Voortgang: 0/39672466 (0%)
Voortgang: 500000/39672466 (1%)
Voortgang: 1000000/39672466 (2%)
Voortgang: 1500000/39672466 (3%)
Voortgang: 2000000/39672466 (5%)
Voortgang: 2500000/39672466 (6%)
Voortgang: 3000000/39672466 (7%)
Voortgang: 3500000/39672466 (8%)
Voortgang: 4000000/39672466 (10%)
Voortgang: 4500000/39672466 (11%)
Voortgang: 5000000/39672466 (12%)
Voortgang: 5500000/39672466 (13%)
Voortgang: 6000000/39672466 (15%)
Voortgang: 6500000/39672466 (16%)
Voortgang: 7000000/39672466 (17%)
Voortgang: 7500000/39672466 (18%)
Voortgang: 8000000/39672466 (20%)
Voortgang: 8500000/39672466 (21%)
Voortgang: 9000000/39672466 (22%)
Voortgang: 9500000/39672466 (23%)
Voortgang: 10000000/39672466 (25%)
Voortgang: 10500000/39672466 (26%)
Voortgang: 11000